# 13 — EfficientNet-B3 Grad-CAM

**Purpose:** Visualize pixels influencing predicted classes for real held-out examples.

**Required inputs:** Selected best_model.pth, test_predictions.csv, grad-cam package.

**Saved outputs:** outputs/classifier/gradcam/ images and summary.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

## 2. Load checkpoint and test categories

Correct blast/normal and FP/FN categories are drawn from actual test predictions.

In [ ]:
import pandas as pd
from src.classifier_config import latest_classifier_run
checkpoint = latest_classifier_run(cfg) / "best_model.pth"
predictions = out / "paper_results" / "test_predictions.csv"
assert predictions.is_file(), "Run notebook 12 first"
frame = pd.read_csv(predictions)
display(pd.crosstab(frame.class_id, frame.predicted_class_id, rownames=["Actual"], colnames=["Predicted"]))

## 3. Generate spatial CAMs

The final feature block has spatial maps. CAM targets the predicted logit and requires gradients in eval mode.

In [ ]:
from src.classifier_gradcam import generate_gradcam
summary = generate_gradcam(checkpoint, predictions, out / "gradcam")
print(summary["category_counts"])
print("Generated examples:", len(summary["generated"]))

## 4. Review and validate saved figure

An absent error category is reported, never fabricated. CAM does not prove morphology.

In [ ]:
from IPython.display import Image as DisplayImage, display
grid = out / "gradcam" / "gradcam_publication_grid.png"
if grid.is_file(): display(DisplayImage(filename=str(grid)))
assert (out / "gradcam" / "gradcam_summary.json").is_file()
print("PASS — Grad-CAM saved; interpret with domain experts")

## Next

Open **14_Classification_Paper_Results.ipynb** for evidence-based reporting.